# Chapter 8 Lab — RAG Pipelines

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 8 — RAG Pipelines  
**Goal:** Build a complete Retrieval-Augmented Generation pipeline from scratch — loading documents, chunking, embedding, storing, retrieving, and generating grounded answers.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Why RAG — hallucination vs grounded answers |
| 2 | Document Loading |
| 3 | Chunking Strategies |
| 4 | Embedding Models |
| 5 | Vector Databases (ChromaDB) |
| 6 | Retrieval Strategies |
| 7 | Re-ranking |
| 8 | Augmented Generation — full RAG pipeline |
| 9 | Advanced Patterns |
| 10 | Evaluation |
| 11 | Common Failure Modes |
| 12 | Exercises |

> **Prerequisites:** An OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────────
# Run once to install required packages (restart the kernel afterwards).

%pip install openai chromadb langchain langchain-community langchain-text-splitters sentence-transformers python-dotenv numpy --quiet

In [ ]:
import json
import os
import textwrap
from typing import Optional

import chromadb
import numpy as np
import openai
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

client = openai.OpenAI()

# Quick connectivity check
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'ready' in one word."}],
    max_tokens=5,
)
print("API check:", resp.choices[0].message.content)

---
## 1. Why RAG — Hallucination vs Grounded Answers

LLMs generate fluent text, but they **hallucinate** when asked about information outside their training data or when they confuse facts. RAG solves this by **retrieving** relevant documents first and feeding them into the prompt so the model can **ground** its answer in real sources.

In [ ]:
# ── Sample knowledge base ────────────────────────────────────────────────────────
# These documents are the "ground truth" the LLM does NOT have in training.

COMPANY_DOCS = [
    {
        "id": "policy-001",
        "text": (
            "Acme Corp Remote Work Policy (Effective January 2026): "
            "All full-time employees may work remotely up to 3 days per week. "
            "Remote work requires manager approval submitted through the HR portal "
            "at least 48 hours in advance. Employees must be available on Slack "
            "during core hours (10 AM – 3 PM local time). Equipment stipend: $1,500/year."
        ),
    },
    {
        "id": "policy-002",
        "text": (
            "Acme Corp PTO Policy (Effective January 2026): "
            "Full-time employees receive 20 days of paid time off per year, "
            "accruing at 1.67 days per month. Unused PTO carries over up to a "
            "maximum of 5 days. PTO requests must be submitted at least 2 weeks "
            "in advance for absences longer than 3 consecutive days."
        ),
    },
    {
        "id": "policy-003",
        "text": (
            "Acme Corp Expense Reimbursement Policy (Effective March 2026): "
            "Business expenses over $50 require pre-approval from a department head. "
            "Receipts must be uploaded to the expense portal within 30 days. "
            "Meals during business travel are reimbursed up to $75/day. "
            "International travel requires VP-level approval."
        ),
    },
    {
        "id": "product-001",
        "text": (
            "AcmeDB v4.2 Release Notes (February 2026): "
            "New features include vector search with HNSW indexing, "
            "automatic sharding for collections over 10M documents, "
            "and a new Python SDK with async support. Breaking change: "
            "the legacy REST API v1 endpoints are removed. "
            "Migration guide available at docs.acme.io/migrate-v4."
        ),
    },
    {
        "id": "product-002",
        "text": (
            "AcmeDB v4.2 Performance Benchmarks: "
            "Vector search latency: p50 = 2ms, p99 = 12ms on 1M vectors (768 dims). "
            "Write throughput: 50,000 docs/sec with batched inserts. "
            "Memory usage: ~1.2 GB per million 768-dim vectors with HNSW index."
        ),
    },
    {
        "id": "support-001",
        "text": (
            "Acme Corp Support Escalation Procedure: "
            "Tier 1 handles basic account and billing issues (target: 4 hour response). "
            "Tier 2 handles technical product issues (target: 8 hour response). "
            "Tier 3 handles critical outages and security incidents (target: 1 hour response). "
            "All Tier 3 escalations trigger an automatic page to the on-call SRE team."
        ),
    },
]

print(f"Loaded {len(COMPANY_DOCS)} sample documents.")

In [ ]:
# ── Without RAG: the model hallucinates ──────────────────────────────────

question = "How many days of PTO do Acme Corp employees get, and how many can carry over?"

no_rag_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an HR assistant for Acme Corp."},
        {"role": "user", "content": question},
    ],
)

print("WITHOUT RAG (likely hallucinated):")
print(textwrap.fill(no_rag_response.choices[0].message.content, width=80))
print()
print("Ground truth: 20 days/year, max 5 days carry over.")

In [ ]:
# ── With RAG: grounded answer ──────────────────────────────────────────
# We manually inject the relevant document into the prompt.

context = COMPANY_DOCS[1]["text"]  # The PTO policy

rag_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are an HR assistant for Acme Corp. Answer questions based ONLY "
                "on the provided context. If the context does not contain the answer, "
                "say 'I don't have that information.'\n\n"
                f"Context:\n{context}"
            ),
        },
        {"role": "user", "content": question},
    ],
)

print("WITH RAG (grounded):")
print(textwrap.fill(rag_response.choices[0].message.content, width=80))

> **Key insight:** The RAG answer matches the ground truth because the model reads the retrieved document rather than guessing. The rest of this lab automates the retrieval step.

---
## 2. Document Loading

RAG pipelines need to ingest documents from many sources. Here we demonstrate loading from plain text, simulated PDF content, and web pages.

In [ ]:
# ── 2a. Plain text loading ──────────────────────────────────────────────────

def load_text_documents(texts: list[dict]) -> list[dict]:
    """Normalize raw text docs into a standard format."""
    docs = []
    for t in texts:
        docs.append({
            "id": t["id"],
            "text": t["text"],
            "source": "inline",
            "type": "text",
        })
    return docs


text_docs = load_text_documents(COMPANY_DOCS)
print(f"Loaded {len(text_docs)} text documents.")
print(f"Sample: {text_docs[0]['text'][:100]}...")

In [ ]:
# ── 2b. Simulated PDF loading ───────────────────────────────────────────────
# In production you would use PyPDF, pdfplumber, or Unstructured.
# Here we simulate the output format.

SIMULATED_PDF_PAGES = [
    {
        "page": 1,
        "text": (
            "Acme Corp Employee Handbook - Table of Contents\n"
            "Chapter 1: Company Overview ..... 3\n"
            "Chapter 2: Remote Work Policy ..... 8\n"
            "Chapter 3: Benefits & PTO ..... 15\n"
            "Chapter 4: Expense Policy ..... 22"
        ),
    },
    {
        "page": 2,
        "text": (
            "Chapter 1: Company Overview\n\n"
            "Acme Corp was founded in 2019 with a mission to build developer tools "
            "that simplify data infrastructure. Headquartered in San Francisco, "
            "the company has grown to 850 employees across 12 countries. Our flagship "
            "product, AcmeDB, serves over 5,000 enterprise customers."
        ),
    },
]


def load_pdf_pages(pages: list[dict], source: str = "handbook.pdf") -> list[dict]:
    """Convert page-level PDF output to standard document format."""
    docs = []
    for p in pages:
        docs.append({
            "id": f"{source}-p{p['page']}",
            "text": p["text"],
            "source": source,
            "type": "pdf",
            "page": p["page"],
        })
    return docs


pdf_docs = load_pdf_pages(SIMULATED_PDF_PAGES)
print(f"Loaded {len(pdf_docs)} PDF pages.")
for d in pdf_docs:
    print(f"  {d['id']}: {d['text'][:60]}...")

In [ ]:
# ── 2c. Web page loading (using LangChain's WebBaseLoader as example) ───────
# We simulate the result to avoid network dependencies during the lab.

SIMULATED_WEB_CONTENT = {
    "url": "https://blog.acme.io/acmedb-v4-2-announcement",
    "text": (
        "Announcing AcmeDB v4.2 — Vector Search Goes Production-Ready\n\n"
        "We are thrilled to announce AcmeDB v4.2, our biggest release yet. "
        "This release brings native vector search powered by HNSW indexing, "
        "making it easy to build RAG pipelines, recommendation engines, and "
        "semantic search directly inside your existing AcmeDB deployment.\n\n"
        "Key highlights:\n"
        "- Vector search with sub-5ms p50 latency at million-scale\n"
        "- Automatic sharding for large collections\n"
        "- New Python SDK with full async/await support\n"
        "- Deprecated: REST API v1 (removed in this release)\n\n"
        "Read the full migration guide at docs.acme.io/migrate-v4."
    ),
}


def load_web_page(page: dict) -> dict:
    """Convert a web scrape result to standard document format."""
    return {
        "id": page["url"],
        "text": page["text"],
        "source": page["url"],
        "type": "web",
    }


web_doc = load_web_page(SIMULATED_WEB_CONTENT)
print(f"Loaded web page: {web_doc['source']}")
print(f"Content preview: {web_doc['text'][:120]}...")

In [ ]:
# ── Combine all documents into a unified corpus ─────────────────────────

ALL_DOCS = text_docs + pdf_docs + [web_doc]
print(f"Total documents in corpus: {len(ALL_DOCS)}")
for d in ALL_DOCS:
    print(f"  [{d['type']:4s}] {d['id']}: {len(d['text'])} chars")

---
## 3. Chunking Strategies

Documents are often too long to embed as a single unit. **Chunking** splits them into smaller pieces. The strategy you choose significantly affects retrieval quality.

| Strategy | How it works | Pros | Cons |
|----------|-------------|------|------|
| Fixed-size | Split every N characters | Simple, predictable | Cuts mid-sentence |
| Recursive | Split on separators (\n\n, \n, . , space) | Respects structure | Needs tuning |
| Semantic | Group sentences by embedding similarity | Best coherence | Slower, more complex |

In [ ]:
# ── 3a. Fixed-size chunking ────────────────────────────────────────────────

def fixed_size_chunk(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """Split text into fixed-size chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


sample_text = COMPANY_DOCS[0]["text"]
fixed_chunks = fixed_size_chunk(sample_text, chunk_size=150, overlap=30)

print(f"Original text ({len(sample_text)} chars) -> {len(fixed_chunks)} chunks\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"Chunk {i} ({len(chunk)} chars): {chunk!r}\n")

In [ ]:
# ── 3b. Recursive chunking (using LangChain) ────────────────────────────

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)

recursive_chunks = recursive_splitter.split_text(sample_text)

print(f"Recursive splitter -> {len(recursive_chunks)} chunks\n")
for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i} ({len(chunk)} chars): {chunk!r}\n")

In [ ]:
# ── 3c. Semantic chunking (simplified) ───────────────────────────────────
# Idea: split into sentences, embed each one, then group consecutive
# sentences whose embeddings are similar (cosine similarity > threshold).

import re


def split_sentences(text: str) -> list[str]:
    """Naive sentence splitter."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def get_embeddings(texts: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    """Get embeddings from OpenAI API."""
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a_arr, b_arr = np.array(a), np.array(b)
    return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))


def semantic_chunk(text: str, threshold: float = 0.75) -> list[str]:
    """Group consecutive sentences by embedding similarity."""
    sentences = split_sentences(text)
    if len(sentences) <= 1:
        return sentences

    embeddings = get_embeddings(sentences)

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(embeddings[i - 1], embeddings[i])
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


semantic_chunks = semantic_chunk(sample_text, threshold=0.7)

print(f"Semantic splitter -> {len(semantic_chunks)} chunks\n")
for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i} ({len(chunk)} chars): {chunk!r}\n")

In [ ]:
# ── Comparison summary ────────────────────────────────────────────────────

print("Chunking comparison on the same document:")
print(f"  Fixed-size:  {len(fixed_chunks)} chunks")
print(f"  Recursive:   {len(recursive_chunks)} chunks")
print(f"  Semantic:    {len(semantic_chunks)} chunks")
print()
print("For the rest of this lab we use recursive chunking — a solid default.")

---
## 4. Embedding Models

Embeddings convert text into dense vectors that capture semantic meaning. Similar texts produce similar vectors.

In [ ]:
# ── 4a. Compare OpenAI embedding models ──────────────────────────────────

test_sentences = [
    "How many vacation days do employees get?",     # query
    "Employees receive 20 days of paid time off.",  # relevant
    "The database supports vector search indexing.", # irrelevant
]

for model_name in ["text-embedding-3-small", "text-embedding-3-large"]:
    embs = get_embeddings(test_sentences, model=model_name)
    sim_relevant = cosine_similarity(embs[0], embs[1])
    sim_irrelevant = cosine_similarity(embs[0], embs[2])
    print(f"Model: {model_name}")
    print(f"  Dims: {len(embs[0])}")
    print(f"  Query vs Relevant:   {sim_relevant:.4f}")
    print(f"  Query vs Irrelevant: {sim_irrelevant:.4f}")
    print(f"  Separation gap:      {sim_relevant - sim_irrelevant:.4f}")
    print()

In [ ]:
# ── 4b. Visualize embeddings in 2D ───────────────────────────────────────
# We embed all our documents + a few queries, then use PCA to project to 2D.

from sklearn.decomposition import PCA
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

viz_texts = [d["text"][:300] for d in COMPANY_DOCS]
viz_labels = [d["id"] for d in COMPANY_DOCS]

queries = [
    "What is the remote work policy?",
    "How fast is vector search?",
    "How do I submit expenses?",
]
viz_texts.extend(queries)
viz_labels.extend([f"Q: {q[:30]}" for q in queries])

all_embs = get_embeddings(viz_texts)
emb_matrix = np.array(all_embs)

pca = PCA(n_components=2)
coords = pca.fit_transform(emb_matrix)

fig, ax = plt.subplots(figsize=(10, 7))
n_docs = len(COMPANY_DOCS)

# Plot documents
ax.scatter(coords[:n_docs, 0], coords[:n_docs, 1], c="steelblue", s=100, label="Documents")
for i in range(n_docs):
    ax.annotate(viz_labels[i], (coords[i, 0], coords[i, 1]), fontsize=8, ha="left")

# Plot queries
ax.scatter(coords[n_docs:, 0], coords[n_docs:, 1], c="tomato", s=100, marker="^", label="Queries")
for i in range(n_docs, len(viz_labels)):
    ax.annotate(viz_labels[i], (coords[i, 0], coords[i, 1]), fontsize=8, ha="left")

ax.set_title("Document & Query Embeddings (PCA 2D Projection)")
ax.legend()
plt.tight_layout()
plt.show()
print("Queries cluster near their relevant documents.")

---
## 5. Vector Databases — ChromaDB

A vector database stores embeddings and supports efficient similarity search. ChromaDB is a lightweight, open-source option perfect for prototyping.

In [ ]:
# ── 5a. Initialize ChromaDB and create a collection ─────────────────────

chroma_client = chromadb.Client()  # in-memory client

# Delete if exists (for re-runs)
try:
    chroma_client.delete_collection("acme_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="acme_docs",
    metadata={"hnsw:space": "cosine"},  # cosine similarity
)

print(f"Collection created: {collection.name}")
print(f"Distance metric: cosine")

In [ ]:
# ── 5b. Chunk and embed all documents, then insert (CREATE) ──────────────

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

all_chunks = []
all_ids = []
all_metadatas = []

for doc in ALL_DOCS:
    chunks = splitter.split_text(doc["text"])
    for j, chunk in enumerate(chunks):
        chunk_id = f"{doc['id']}_chunk{j}"
        all_chunks.append(chunk)
        all_ids.append(chunk_id)
        all_metadatas.append({
            "source_id": doc["id"],
            "source_type": doc["type"],
            "chunk_index": j,
        })

# Get embeddings from OpenAI
all_embeddings = get_embeddings(all_chunks)

# Insert into ChromaDB
collection.add(
    ids=all_ids,
    embeddings=all_embeddings,
    documents=all_chunks,
    metadatas=all_metadatas,
)

print(f"Inserted {collection.count()} chunks into ChromaDB.")
print(f"\nSample IDs: {all_ids[:5]}")

In [ ]:
# ── 5c. READ — query the collection ──────────────────────────────────────

query = "What is the remote work policy?"
query_emb = get_embeddings([query])

results = collection.query(
    query_embeddings=query_emb,
    n_results=3,
    include=["documents", "distances", "metadatas"],
)

print(f"Query: {query}\n")
for i in range(len(results["ids"][0])):
    print(f"  Rank {i+1} (distance: {results['distances'][0][i]:.4f}):")
    print(f"    ID: {results['ids'][0][i]}")
    print(f"    Text: {results['documents'][0][i][:120]}...")
    print()

In [ ]:
# ── 5d. UPDATE — modify a document ──────────────────────────────────────

updated_text = (
    "Acme Corp Remote Work Policy (UPDATED March 2026): "
    "All full-time employees may work remotely up to 4 days per week. "
    "Remote work requires manager approval via the HR portal "
    "at least 24 hours in advance. Equipment stipend: $2,000/year."
)
updated_emb = get_embeddings([updated_text])

collection.update(
    ids=["policy-001_chunk0"],
    embeddings=updated_emb,
    documents=[updated_text],
    metadatas=[{"source_id": "policy-001", "source_type": "text", "chunk_index": 0, "version": "v2"}],
)

# Verify the update
result = collection.get(ids=["policy-001_chunk0"], include=["documents", "metadatas"])
print("Updated document:")
print(f"  Text: {result['documents'][0][:120]}...")
print(f"  Metadata: {result['metadatas'][0]}")

In [ ]:
# ── 5e. DELETE — remove a document ──────────────────────────────────────

count_before = collection.count()

# Delete a specific chunk
collection.delete(ids=["handbook.pdf-p1_chunk0"])

count_after = collection.count()
print(f"Documents before delete: {count_before}")
print(f"Documents after delete:  {count_after}")
print(f"Deleted {count_before - count_after} chunk(s).")

---
## 6. Retrieval Strategies

Not all retrieval is simple nearest-neighbor search. Different strategies optimize for different goals.

In [ ]:
# ── 6a. Basic similarity search ─────────────────────────────────────────

def similarity_search(query: str, n: int = 3) -> list[dict]:
    """Return the top-n most similar chunks to the query."""
    query_emb = get_embeddings([query])
    results = collection.query(
        query_embeddings=query_emb,
        n_results=n,
        include=["documents", "distances", "metadatas"],
    )
    return [
        {
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "distance": results["distances"][0][i],
            "metadata": results["metadatas"][0][i],
        }
        for i in range(len(results["ids"][0]))
    ]


query = "How do I get reimbursed for a business dinner?"
sim_results = similarity_search(query, n=3)

print(f"Similarity search for: '{query}'\n")
for r in sim_results:
    print(f"  [{r['distance']:.4f}] {r['id']}")
    print(f"    {r['text'][:120]}...\n")

In [ ]:
# ── 6b. Maximal Marginal Relevance (MMR) ───────────────────────────────
# MMR balances relevance with diversity: it penalizes results that are
# too similar to already-selected results.

def mmr_search(
    query: str, n: int = 3, candidates: int = 10, lambda_param: float = 0.5
) -> list[dict]:
    """Maximal Marginal Relevance search."""
    query_emb = get_embeddings([query])[0]

    # Fetch more candidates than needed
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=candidates,
        include=["documents", "distances", "metadatas", "embeddings"],
    )

    candidate_embs = results["embeddings"][0]
    candidate_docs = results["documents"][0]
    candidate_ids = results["ids"][0]
    candidate_metas = results["metadatas"][0]
    candidate_dists = results["distances"][0]

    selected_indices = []
    remaining = list(range(len(candidate_docs)))

    for _ in range(min(n, len(remaining))):
        best_score = -float("inf")
        best_idx = remaining[0]

        for idx in remaining:
            # Relevance to query (convert distance to similarity)
            relevance = 1 - candidate_dists[idx]

            # Max similarity to already-selected
            if selected_indices:
                max_sim_to_selected = max(
                    cosine_similarity(candidate_embs[idx], candidate_embs[s])
                    for s in selected_indices
                )
            else:
                max_sim_to_selected = 0

            # MMR score
            score = lambda_param * relevance - (1 - lambda_param) * max_sim_to_selected

            if score > best_score:
                best_score = score
                best_idx = idx

        selected_indices.append(best_idx)
        remaining.remove(best_idx)

    return [
        {
            "id": candidate_ids[i],
            "text": candidate_docs[i],
            "distance": candidate_dists[i],
            "metadata": candidate_metas[i],
        }
        for i in selected_indices
    ]


mmr_results = mmr_search(query, n=3, lambda_param=0.5)

print(f"MMR search for: '{query}'\n")
for r in mmr_results:
    print(f"  [{r['distance']:.4f}] {r['id']}")
    print(f"    {r['text'][:120]}...\n")

print("Note: MMR results are more diverse than pure similarity search.")

In [ ]:
# ── 6c. Hybrid search (keyword + semantic) ─────────────────────────────
# Combine BM25-style keyword matching with vector similarity.

def keyword_score(query: str, text: str) -> float:
    """Simple keyword overlap score (Jaccard-like)."""
    query_tokens = set(query.lower().split())
    text_tokens = set(text.lower().split())
    if not query_tokens:
        return 0.0
    return len(query_tokens & text_tokens) / len(query_tokens)


def hybrid_search(
    query: str, n: int = 3, alpha: float = 0.7
) -> list[dict]:
    """
    Hybrid search combining semantic similarity and keyword matching.
    alpha: weight for semantic score (1-alpha for keyword score).
    """
    # Semantic search
    query_emb = get_embeddings([query])
    results = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        include=["documents", "distances", "metadatas"],
    )

    scored = []
    for i in range(len(results["ids"][0])):
        semantic_score = 1 - results["distances"][0][i]  # cosine distance -> similarity
        kw_score = keyword_score(query, results["documents"][0][i])
        combined = alpha * semantic_score + (1 - alpha) * kw_score
        scored.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "semantic_score": semantic_score,
            "keyword_score": kw_score,
            "combined_score": combined,
            "metadata": results["metadatas"][0][i],
        })

    scored.sort(key=lambda x: x["combined_score"], reverse=True)
    return scored[:n]


hybrid_results = hybrid_search("AcmeDB REST API migration", n=3)

print("Hybrid search for: 'AcmeDB REST API migration'\n")
for r in hybrid_results:
    print(f"  [{r['combined_score']:.4f}] {r['id']}")
    print(f"    Semantic: {r['semantic_score']:.3f}  Keyword: {r['keyword_score']:.3f}")
    print(f"    {r['text'][:120]}...\n")

---
## 7. Re-ranking

After initial retrieval, a **re-ranker** reorders results using a more expensive but more accurate model. This is a two-stage pattern: fast retrieval (vector search) followed by precise re-ranking.

In [ ]:
# ── 7a. LLM-based re-ranker ──────────────────────────────────────────────
# Ask the LLM to score each candidate's relevance to the query.

def llm_rerank(query: str, candidates: list[dict], top_n: int = 3) -> list[dict]:
    """Use an LLM to re-rank retrieved candidates by relevance."""
    reranked = []

    for candidate in candidates:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a relevance scoring system. Rate how relevant the "
                        "passage is to the query on a scale of 0 to 10. "
                        "Respond with ONLY a number, nothing else."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Query: {query}\n\nPassage: {candidate['text']}",
                },
            ],
            max_tokens=5,
            temperature=0,
        )
        try:
            score = float(response.choices[0].message.content.strip())
        except ValueError:
            score = 0.0

        reranked.append({**candidate, "rerank_score": score})

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_n]


# First retrieve broadly, then re-rank
query = "What happens during a critical outage?"
initial_results = similarity_search(query, n=6)

print("Before re-ranking:")
for i, r in enumerate(initial_results):
    print(f"  {i+1}. [{r['distance']:.4f}] {r['id']}: {r['text'][:80]}...")

reranked = llm_rerank(query, initial_results, top_n=3)

print("\nAfter LLM re-ranking:")
for i, r in enumerate(reranked):
    print(f"  {i+1}. [score: {r['rerank_score']:.0f}/10] {r['id']}: {r['text'][:80]}...")

In [ ]:
# ── 7b. Cross-encoder style re-ranker (embedding-based) ─────────────────
# A lighter alternative: re-score using the full query+passage embedding.

def embedding_rerank(query: str, candidates: list[dict], top_n: int = 3) -> list[dict]:
    """Re-rank by embedding the query-passage pair as a single string."""
    pairs = [f"Query: {query} Passage: {c['text']}" for c in candidates]
    pair_embs = get_embeddings(pairs)
    query_emb = get_embeddings([f"Query: {query} Passage: "])[0]

    scored = []
    for i, candidate in enumerate(candidates):
        sim = cosine_similarity(query_emb, pair_embs[i])
        scored.append({**candidate, "rerank_score": sim})

    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_n]


emb_reranked = embedding_rerank(query, initial_results, top_n=3)

print("Embedding-based re-ranking:")
for i, r in enumerate(emb_reranked):
    print(f"  {i+1}. [score: {r['rerank_score']:.4f}] {r['id']}: {r['text'][:80]}...")

---
## 8. Augmented Generation — The Full RAG Pipeline

Now we wire everything together: **query → retrieve → (re-rank) → generate**.

In [ ]:
# ── 8a. Complete RAG pipeline ───────────────────────────────────────────

def rag_pipeline(
    query: str,
    n_retrieve: int = 5,
    n_rerank: int = 3,
    use_reranking: bool = True,
    model: str = "gpt-4o-mini",
) -> dict:
    """
    Full RAG pipeline: retrieve -> re-rank -> generate.

    Returns dict with 'answer', 'sources', and 'context'.
    """
    # Step 1: Retrieve
    candidates = similarity_search(query, n=n_retrieve)

    # Step 2: Re-rank (optional)
    if use_reranking and len(candidates) > n_rerank:
        top_docs = llm_rerank(query, candidates, top_n=n_rerank)
    else:
        top_docs = candidates[:n_rerank]

    # Step 3: Build context
    context_parts = []
    for i, doc in enumerate(top_docs):
        context_parts.append(f"[Source {i+1}: {doc['id']}]\n{doc['text']}")
    context = "\n\n".join(context_parts)

    # Step 4: Generate
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Answer the user's question based ONLY "
                    "on the provided context. Cite your sources using [Source N] notation. "
                    "If the context does not contain enough information, say so explicitly.\n\n"
                    f"Context:\n{context}"
                ),
            },
            {"role": "user", "content": query},
        ],
        temperature=0,
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [doc["id"] for doc in top_docs],
        "context": context,
    }


print("RAG pipeline function defined.")

In [ ]:
# ── 8b. Test the pipeline ───────────────────────────────────────────────

test_queries = [
    "How many days can I work from home per week?",
    "What are the performance benchmarks for AcmeDB vector search?",
    "What is the meal reimbursement limit during business travel?",
    "How quickly should a critical outage be responded to?",
]

for q in test_queries:
    result = rag_pipeline(q, use_reranking=False)  # skip reranking for speed
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources: {result['sources']}")
    print("-" * 80)
    print()

---
## 9. Advanced Patterns

### 9a. Parent-Child Retrieval
Retrieve on small chunks (for precision) but pass the larger parent chunk to the LLM (for context).

### 9b. HyDE (Hypothetical Document Embeddings)
Generate a hypothetical answer first, embed that, and use it as the search query. This bridges the vocabulary gap between questions and documents.

In [ ]:
# ── 9a. Parent-Child Retrieval ─────────────────────────────────────────

# Build a parent-child index: small child chunks for retrieval,
# mapped back to larger parent chunks for generation context.

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)

parent_store = {}  # parent_id -> parent_text
child_to_parent = {}  # child_id -> parent_id

# Create a separate collection for child chunks
try:
    chroma_client.delete_collection("acme_parent_child")
except Exception:
    pass

pc_collection = chroma_client.create_collection(
    name="acme_parent_child",
    metadata={"hnsw:space": "cosine"},
)

child_ids = []
child_texts = []
child_metas = []

for doc in COMPANY_DOCS:
    parents = parent_splitter.split_text(doc["text"])
    for pi, parent_text in enumerate(parents):
        parent_id = f"{doc['id']}_parent{pi}"
        parent_store[parent_id] = parent_text

        children = child_splitter.split_text(parent_text)
        for ci, child_text in enumerate(children):
            child_id = f"{parent_id}_child{ci}"
            child_to_parent[child_id] = parent_id
            child_ids.append(child_id)
            child_texts.append(child_text)
            child_metas.append({"parent_id": parent_id, "source_id": doc["id"]})

child_embeddings = get_embeddings(child_texts)
pc_collection.add(
    ids=child_ids,
    embeddings=child_embeddings,
    documents=child_texts,
    metadatas=child_metas,
)

print(f"Parent chunks: {len(parent_store)}")
print(f"Child chunks:  {pc_collection.count()}")


def parent_child_retrieve(query: str, n: int = 3) -> list[str]:
    """Retrieve on children, return parent texts."""
    query_emb = get_embeddings([query])
    results = pc_collection.query(
        query_embeddings=query_emb,
        n_results=n,
        include=["metadatas"],
    )
    seen_parents = set()
    parent_texts = []
    for meta in results["metadatas"][0]:
        pid = meta["parent_id"]
        if pid not in seen_parents:
            seen_parents.add(pid)
            parent_texts.append(parent_store[pid])
    return parent_texts


parents = parent_child_retrieve("equipment stipend for remote work")
print("\nParent-child retrieval result:")
for i, p in enumerate(parents):
    print(f"  Parent {i+1}: {p[:150]}...")

In [ ]:
# ── 9b. HyDE (Hypothetical Document Embeddings) ───────────────────────

def hyde_search(query: str, n: int = 3) -> list[dict]:
    """
    HyDE: Generate a hypothetical answer, embed it, and search.
    The hypothetical answer is closer in embedding space to the
    actual document than the question alone.
    """
    # Step 1: Generate hypothetical answer
    hypo_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Write a short paragraph that would be a good answer to the "
                    "following question. Write as if you are quoting from a "
                    "company policy document. Do not hedge or say 'I don't know'."
                ),
            },
            {"role": "user", "content": query},
        ],
        temperature=0,
        max_tokens=150,
    )
    hypothetical = hypo_response.choices[0].message.content
    print(f"  HyDE hypothetical: {hypothetical[:120]}...")

    # Step 2: Embed the hypothetical answer (not the question)
    hyde_emb = get_embeddings([hypothetical])

    # Step 3: Search with the hypothetical embedding
    results = collection.query(
        query_embeddings=hyde_emb,
        n_results=n,
        include=["documents", "distances", "metadatas"],
    )

    return [
        {
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "distance": results["distances"][0][i],
        }
        for i in range(len(results["ids"][0]))
    ]


print("HyDE search for: 'What is the PTO carryover limit?'\n")
hyde_results = hyde_search("What is the PTO carryover limit?", n=3)
print()
for r in hyde_results:
    print(f"  [{r['distance']:.4f}] {r['id']}: {r['text'][:100]}...")

print("\n--- Compare with standard search ---")
std_results = similarity_search("What is the PTO carryover limit?", n=3)
for r in std_results:
    print(f"  [{r['distance']:.4f}] {r['id']}: {r['text'][:100]}...")

---
## 10. Evaluation

How do you know if your RAG pipeline is working well? We measure both **retrieval quality** and **generation quality**.

In [ ]:
# ── 10a. Retrieval metrics ───────────────────────────────────────────────

def precision_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    """What fraction of the top-k retrieved docs are relevant?"""
    top_k = retrieved_ids[:k]
    relevant_in_top_k = sum(1 for rid in top_k if rid in relevant_ids)
    return relevant_in_top_k / k


def recall_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    """What fraction of all relevant docs appear in the top-k?"""
    top_k = retrieved_ids[:k]
    relevant_in_top_k = sum(1 for rid in top_k if rid in relevant_ids)
    return relevant_in_top_k / len(relevant_ids) if relevant_ids else 0.0


def mrr(retrieved_ids: list[str], relevant_ids: set[str]) -> float:
    """Mean Reciprocal Rank: 1/position of the first relevant result."""
    for i, rid in enumerate(retrieved_ids):
        if rid in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0


# Test with a known query-relevance pair
eval_query = "What is the remote work equipment stipend?"
# We know policy-001 chunks are relevant
relevant = {"policy-001_chunk0"}  # the updated remote work policy chunk

eval_results = similarity_search(eval_query, n=5)
retrieved = [r["id"] for r in eval_results]

print(f"Query: {eval_query}")
print(f"Retrieved: {retrieved}")
print(f"Relevant:  {relevant}")
print(f"\nPrecision@1: {precision_at_k(retrieved, relevant, 1):.2f}")
print(f"Precision@3: {precision_at_k(retrieved, relevant, 3):.2f}")
print(f"Recall@3:    {recall_at_k(retrieved, relevant, 3):.2f}")
print(f"MRR:         {mrr(retrieved, relevant):.2f}")

In [ ]:
# ── 10b. Generation quality — faithfulness check ───────────────────────
# Use the LLM itself to judge whether the answer is faithful to the context.

def check_faithfulness(query: str, answer: str, context: str) -> dict:
    """LLM-as-judge: does the answer stay faithful to the context?"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an evaluation judge. Given a query, a context, and an answer, "
                    "determine whether the answer is faithful to the context.\n\n"
                    "Respond in JSON format:\n"
                    '{"faithful": true/false, "reason": "brief explanation"}'
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Query: {query}\n\n"
                    f"Context: {context}\n\n"
                    f"Answer: {answer}"
                ),
            },
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


# Evaluate a good answer
result = rag_pipeline("What is the meal reimbursement limit?", use_reranking=False)
faith = check_faithfulness(
    "What is the meal reimbursement limit?",
    result["answer"],
    result["context"],
)
print("Faithfulness check (good answer):")
print(f"  Faithful: {faith['faithful']}")
print(f"  Reason: {faith['reason']}")

# Evaluate a fabricated answer
faith_bad = check_faithfulness(
    "What is the meal reimbursement limit?",
    "The meal reimbursement limit is $150 per day with no receipt required.",
    result["context"],
)
print("\nFaithfulness check (fabricated answer):")
print(f"  Faithful: {faith_bad['faithful']}")
print(f"  Reason: {faith_bad['reason']}")

In [ ]:
# ── 10c. Run a mini evaluation suite ───────────────────────────────────

eval_set = [
    {
        "query": "How many remote work days per week are allowed?",
        "expected_source": "policy-001",
        "expected_keywords": ["4 days"],  # updated policy
    },
    {
        "query": "What is the p99 latency for AcmeDB vector search?",
        "expected_source": "product-002",
        "expected_keywords": ["12ms"],
    },
    {
        "query": "How much PTO carries over?",
        "expected_source": "policy-002",
        "expected_keywords": ["5 days"],
    },
]

print("Mini evaluation suite:\n")
for test in eval_set:
    result = rag_pipeline(test["query"], use_reranking=False)
    answer = result["answer"]

    # Check source retrieval
    source_hit = any(test["expected_source"] in s for s in result["sources"])

    # Check keyword presence in answer
    keyword_hits = [
        kw for kw in test["expected_keywords"]
        if kw.lower() in answer.lower()
    ]

    status = "PASS" if source_hit and keyword_hits else "FAIL"
    print(f"  [{status}] {test['query']}")
    print(f"    Answer: {answer[:120]}...")
    print(f"    Source hit: {source_hit}, Keywords found: {keyword_hits}")
    print()

---
## 11. Common Failure Modes

RAG pipelines fail in predictable ways. Understanding these patterns helps you diagnose and fix issues quickly.

| Failure Mode | Symptom | Root Cause |
|-------------|---------|------------|
| Retrieval miss | Correct doc not in top-k | Bad chunking, wrong embedding model |
| Context overflow | Answer ignores some sources | Too many/long chunks in prompt |
| Vocabulary mismatch | Query uses different terms than docs | Need HyDE or query expansion |
| Stale data | Answer cites outdated info | Index not refreshed |
| Hallucination despite context | Answer adds facts not in context | Weak system prompt, high temperature |

In [ ]:
# ── 11a. Failure: Retrieval miss due to vocabulary mismatch ──────────────

# The user asks about "vacation" but the document says "PTO"
mismatch_query = "How many vacation days do new hires get?"
mismatch_results = similarity_search(mismatch_query, n=3)

print(f"Query: {mismatch_query}\n")
print("Retrieved documents:")
for r in mismatch_results:
    print(f"  [{r['distance']:.4f}] {r['id']}: {r['text'][:100]}...")

# Check if the PTO policy was retrieved
pto_found = any("policy-002" in r["id"] for r in mismatch_results)
print(f"\nPTO policy found in top-3: {pto_found}")
if not pto_found:
    print("FAILURE: Vocabulary mismatch - 'vacation' vs 'PTO'.")
    print("FIX: Use HyDE, add synonyms to docs, or use hybrid search.")
else:
    print("Embedding model handled synonym well.")

In [ ]:
# ── 11b. Failure: Hallucination despite having context ─────────────────

# A weak system prompt + high temperature can cause the model to
# add information not present in the context.

context_text = COMPANY_DOCS[2]["text"]  # expense policy

# Weak prompt (no grounding instruction)
weak_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Help the user. Context: {context_text}"},
        {"role": "user", "content": "What's our policy on expensing team dinners and entertainment?"},
    ],
    temperature=1.0,  # high temperature = more creative/hallucinated
)

print("Weak prompt + high temperature:")
print(textwrap.fill(weak_response.choices[0].message.content, width=80))
print()
print("NOTE: The context says nothing about 'team dinners' or 'entertainment'.")
print("The model likely fabricated details because the prompt did not constrain it.")
print()

# Strong prompt (explicit grounding)
strong_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "Answer ONLY based on the context below. If the context does not "
                "contain the answer, respond: 'This is not covered in the policy.'\n\n"
                f"Context: {context_text}"
            ),
        },
        {"role": "user", "content": "What's our policy on expensing team dinners and entertainment?"},
    ],
    temperature=0,
)

print("Strong prompt + temperature=0:")
print(textwrap.fill(strong_response.choices[0].message.content, width=80))

In [ ]:
# ── 11c. Failure: Chunking splits critical information ─────────────────

# Demonstrate how bad chunk boundaries destroy retrieval.

problematic_text = (
    "Security Incident Response: In case of a data breach, immediately notify "
    "the CISO at security@acme.io. The response team must be assembled within "
    "30 minutes. All affected systems must be isolated. Customer notification "
    "must happen within 72 hours per GDPR requirements."
)

# Tiny chunk size splits the instruction across chunks
bad_chunks = fixed_size_chunk(problematic_text, chunk_size=80, overlap=0)

print("Bad chunking (80 chars, no overlap):")
for i, chunk in enumerate(bad_chunks):
    print(f"  Chunk {i}: {chunk!r}")

print("\nProblem: '72 hours per GDPR' is separated from 'Customer notification'.")
print("A query about GDPR notification timeline might miss the deadline.")

# Better: larger chunks with overlap
good_chunks = fixed_size_chunk(problematic_text, chunk_size=200, overlap=50)
print(f"\nBetter chunking (200 chars, 50 overlap): {len(good_chunks)} chunks")
for i, chunk in enumerate(good_chunks):
    print(f"  Chunk {i}: {chunk!r}")

---
## 12. Exercises

### Exercise 1: Conceptual Questions

Answer these in a markdown cell below:

1. Why does chunk overlap matter? What happens if overlap is 0? What if it equals chunk size?
2. When would you choose MMR over pure similarity search? Give a concrete example.
3. Explain why HyDE works. In what scenarios would it hurt rather than help?
4. A user reports that the RAG system gives correct answers for short factual questions but fails on multi-hop questions like "Compare the remote work and expense policies." Diagnose the likely cause and propose a fix.

*Your answers here...*

### Exercise 2: Coding — Build an End-to-End RAG System

Build a RAG chatbot for a fictional **restaurant menu + review** dataset. Requirements:

1. Create at least 8 documents covering menu items (with prices, ingredients, dietary info) and customer reviews.
2. Implement chunking with `RecursiveCharacterTextSplitter`.
3. Store in ChromaDB with metadata (document type, category).
4. Implement **metadata filtering** (e.g., only search reviews, or only vegetarian items).
5. Build a `chat()` function that maintains conversation history and answers questions like:
   - "What vegetarian options do you have under $15?"
   - "What did customers say about the pasta?"
   - "Recommend a dish for someone with a nut allergy."

In [ ]:
# Exercise 2: Your code here
# Starter structure:

RESTAURANT_DOCS = [
    # TODO: Add 8+ documents (menu items + reviews)
    # Example:
    # {"id": "menu-001", "text": "...", "type": "menu", "category": "pasta"},
    # {"id": "review-001", "text": "...", "type": "review", "rating": 5},
]


def build_restaurant_index(docs):
    """Chunk, embed, and store restaurant docs in ChromaDB."""
    # TODO: implement
    pass


def restaurant_search(query, doc_type=None, n=3):
    """Search with optional metadata filtering."""
    # TODO: implement using collection.query(..., where={"type": doc_type})
    pass


def chat(user_message, history=None):
    """RAG chatbot with conversation history."""
    # TODO: implement
    pass

### Exercise 3: Design — Multi-Source RAG Architecture

You are designing a RAG system for a **financial services company** that needs to answer questions using:

- **Internal policy documents** (PDFs, updated quarterly)
- **Real-time market data** (APIs, changes every second)
- **Customer support tickets** (database, 100K+ records)
- **Regulatory filings** (SEC EDGAR, structured + unstructured)

In a markdown cell below, design the system by addressing:

1. **Data pipeline**: How do you ingest, chunk, and keep each source up to date?
2. **Retrieval strategy**: Do you use one vector store or multiple? How do you route queries to the right source?
3. **Freshness**: How do you handle the fact that market data changes every second but policies change quarterly?
4. **Evaluation**: What metrics would you track in production? How would you detect retrieval degradation?
5. **Failure handling**: What happens when the vector DB is down? When the embedding API has high latency?

Draw an architecture diagram in ASCII or describe the components and their interactions.

*Your design here...*

---
## Summary

In this lab you built a complete RAG pipeline from first principles:

| Component | What you learned |
|-----------|------------------|
| **Why RAG** | LLMs hallucinate without grounding; RAG fixes this by injecting retrieved context |
| **Document Loading** | Ingesting text, PDFs, and web pages into a unified format |
| **Chunking** | Fixed-size, recursive, and semantic strategies with different trade-offs |
| **Embeddings** | Converting text to vectors; comparing models; visualizing the embedding space |
| **Vector DB** | ChromaDB CRUD operations for storing and querying embeddings |
| **Retrieval** | Similarity search, MMR for diversity, hybrid search for keyword + semantic |
| **Re-ranking** | LLM-based and embedding-based re-ranking for precision |
| **Generation** | Wiring retrieval into a grounded generation pipeline with source citations |
| **Advanced** | Parent-child retrieval for context, HyDE for vocabulary gap bridging |
| **Evaluation** | Precision, recall, MRR for retrieval; faithfulness checking for generation |
| **Failures** | Vocabulary mismatch, bad chunking, weak prompts, and how to diagnose them |

**Next up:** Chapter 9 — Tool Use & Function Calling, where we give agents the ability to act on the world.